# Lab: DBSCAN and HDBSCAN

*In this lab, we will apply density-based clustering algorithms (DBSCAN and HDBSCAN) to museum locations across Canada to help the government determine optimal regional groupings for administrative oversight.*

---

> ### 📝 1. Business Understanding Report (Summary)
> * **Business Context:** Statistics Canada (StatCan) maintains the Open Database of Cultural and Art Facilities (ODCAF), which contains comprehensive information about cultural and art institutions across the country. The Government of Canada is seeking to optimize its administrative structure for museum oversight by assigning regional principals (administrators) to geographically clustered museums.
> * **Business Problem:** The government needs to determine the optimal number of regional clusters for museums across Canada to:
>   1. Assign the appropriate number of regional principals (one principal per cluster)
>   2. Minimize travel and administrative overhead by grouping geographically proximate museums
>   3. Ensure efficient communication and resource sharing between neighboring institutions
>   4. Balance cluster sizes to distribute administrative workload fairly across principals
> * **Project Goals:**
>   1. Discover natural geographic groupings of museums based on spatial proximity
>   2. Recommend an optimal number of regional clusters (and thus regional principals) based on clustering results
>   3. Identify isolated museums that may require special administrative considerations
> * **Success Criteria:**
>   - Clear, interpretable geographic clusters visible on a map of Canada
>   - Reasonable cluster sizes that support equitable distribution of administrative responsibility
>   - Identification of geographic outliers or isolated museums
>   - Evidence-based recommendation on the number of regional principals needed

---

> ### 📝 2. Analytic Approach Report (Summary)
> * **Problem Type:** Unsupervised Learning - Spatial Clustering, with focus on discovering natural geographic groupings based on proximity rather than pre-specifying the number of clusters.
> * **Why Density-Based Clustering?** Unlike K-Means (which requires specifying k clusters in advance and assumes spherical clusters), density-based clustering:
>   - Discovers clusters of arbitrary shapes, ideal for geographic data with irregular distributions
>   - Automatically determines the number of clusters without prior assumptions
>   - Identifies isolated museums (noise points) that may need special handling
>   - Handles varying densities across regions (e.g., densely packed museums in urban areas vs. sparse rural areas)
> * **Algorithms to Compare:**
>   1. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise):**
>       - Core parameters: `eps` (neighborhood radius in degrees) and `min_samples` (minimum museums to form a cluster)
>       - Strength: Simple, interpretable, good for uniform density
>       - Limitation: Struggles with varying densities (may over-connect nearby clusters or fragment sparse regions)
>   2. **HDBSCAN (Hierarchical DBSCAN):**
>       - Core parameter: `min_cluster_size` (minimum museums per cluster)
>       - Strength: Handles varying densities naturally, fewer parameters to tune
>       - Advantage: Better suited for geographic data with mixed urban/rural patterns
> * **Evaluation Metrics:**
>   - **Number of clusters discovered** to inform optimal number of regional principals
>   - **Cluster size distribution** to assess fairness of administrative workload
>   - **Number of noise points** to identify isolated museums requiring special considerations
>   - **Visual inspection on map** to assess geographic sensibility of clusters
>   - **Comparison DBSCAN vs HDBSCAN** to determine which algorithm provides more actionable insights
> * **Deliverable:** Recommendation on optimal number of regional clusters (and thus regional principals) with supporting geographic visualization and cluster analysis

---

> ### 📝 3. Data Requirements Report (Summary)
> * **Data Sources:**
>   - **Museums dataset (ODCAF):** https://www.statcan.gc.ca/en/lode/databases/odcaf (CSV with facility names, types, and coordinates)
>   - **Canada basemap (GeoTIFF):** https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YcUk-ytgrPkmvZAh5bf7zA/Canada.zip
> * **License:** Open Government License - Canada
> * **Dataset Overview:**
>   - Multiple facility types; we filter to **museums only**
>   - Each record includes facility name, type, latitude, longitude
> * **Features to Use:**
>   - **Latitude** (±90°) and **Longitude** (0–360°) are sufficient for geographic proximity and clustering
>   - **Index** for reference; other attributes not needed for clustering
> * **Data Quality Considerations:**
>   - Missing values represented as `'..'`
>   - Coordinates loaded as strings → convert to floats
>   - Remove rows without valid coordinates
> * **Basemap Use:** Download and extract the Canada GeoTIFF for background mapping of clusters.
---

## Stage 4: Data Collection
As usual, we start with importing the necessary libraries and configuring them.

In [68]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import io
import requests
import zipfile
import joblib
from pathlib import Path

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1. Extraction
Let's start with extracting the Canada map.

In [69]:
# URL of the ZIP file on the cloud server
zip_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YcUk-ytgrPkmvZAh5bf7zA/Canada.zip"

# Define and create the raw data directory
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Check if Canada.tif already exists
canada_tif_path = raw_data_dir / "Canada.tif"

if canada_tif_path.exists():
    print(f"File already exists: {canada_tif_path}")
else:
    # Download the ZIP file
    response = requests.get(zip_url, stream=True, timeout=30)
    response.raise_for_status()
    
    # Open the ZIP file in memory
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_name in zip_ref.namelist():
            if file_name.endswith('.tif'):
                target = raw_data_dir / Path(file_name).name
                zip_ref.extract(file_name, raw_data_dir)
                print(f"Downloaded and extracted: {target}")

File already exists: ../data/raw/Canada.tif


Now, let's extract the dataset.

In [70]:
raw_data_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/r-maSj5Yegvw2sJraT15FA/ODCAF-v1-0.csv'

raw_data_path = raw_data_dir / 'ODCAF-v1-0.csv'

if not raw_data_path.exists():
    response = requests.get(raw_data_url, stream=True, timeout=30)
    response.raise_for_status()
    with open(raw_data_path, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded raw data to: {raw_data_path}")
else:
    print(f"Raw data file already exists: {raw_data_path}")

Raw data file already exists: ../data/raw/ODCAF-v1-0.csv


In [71]:
raw_df = pd.read_csv(raw_data_path, encoding = "ISO-8859-1", na_values=['..'])
print(f"Extracted dataset has {raw_df.shape[0]} samples and {raw_df.shape[1]} features, with {raw_df.isnull().values.sum()} missing values out of {raw_df.size}.")
raw_df.head()

Extracted dataset has 7972 samples and 17 features, with 18109 missing values out of 135524.


,Index,Facility_Name,Source_Facility_Type,ODCAF_Facility_Type,Provider,Unit,Street_No,Street_Name,Postal_Code,City,Prov_Terr,Source_Format_Address,CSD_Name,CSDUID,PRUID,Latitude,Longitude
0,1,#Hashtag Gallery,NaN,gallery,toronto,NaN,801,dundas st w,M6J 1V2,toronto,on,801 dundas st w,Toronto,3520005.0,35.0,43.65169472,-79.408033
1,2,'Ksan Historical Village & Museum,historic site-building or park,museum,canadian museums association,NaN,1500,62 hwy,V0J 1Y0,hazelton,bc,1500 hwy 62 hazelton british columbia v0j 1y0 ...,Hazelton,5949022.0,59.0,55.2645508,-127.642812
2,3,'School Days' Museum,community/regional museum,museum,canadian museums association,NaN,427,queen st,E3B 5R6,fredericton,nb,427 queen st fredericton new brunswick e3b 5r6...,Fredericton,1310032.0,13.0,45.963283,-66.641902
3,4,10 Austin Street,built heritage properties,heritage or historic site,moncton,NaN,10,austin st,E1C 1Z6,moncton,nb,10 austin st,Moncton,1307022.0,13.0,46.09247776,-64.780229
4,5,10 Gates Dancing Inc.,arts,miscellaneous,ottawa,NaN,NaN,NaN,NaN,ottawa,on,NaN,Ottawa,3506008.0,35.0,45.40856224,-75.715368


### 4.2. Transformation

In [72]:
# Read the raw data
interim_df = pd.read_csv(raw_data_path, encoding = "ISO-8859-1", na_values=['..'])

In [73]:
# Select museums only
interim_df = interim_df[interim_df['ODCAF_Facility_Type'] == 'museum']

# Only keep the Index, Latitude and Longitude columns  
interim_df = interim_df[['Index', 'Latitude', 'Longitude']]

# Drop rows with missing values
interim_df = interim_df.dropna()

# Convert column names to lower case
interim_df.columns = [col.lower() for col in interim_df.columns]

# Convert latitude to float
interim_df['latitude'] = interim_df['latitude'].str.replace(',', '').astype(float)

In [74]:
# Manually check the minimum and maximum values for all numerical columns for any abnormalities
print("\n--- Min/Max Summary of Numerical Data ---")
display(interim_df.describe().loc[['min', 'max']])


--- Min/Max Summary of Numerical Data ---


,index,latitude,longitude
min,2.0,41.737209,-139.431695
max,9798.0,67.029613,-52.684217


* 📌 Latitude and longitude values are within the expected ranges.

### 4.3. Loading

In [75]:
# Save interim dataframe as a parquet file
interim_data_dir = Path("../data/interim")
interim_data_dir.mkdir(parents=True, exist_ok=True)
interim_data_path = interim_data_dir / "interim_museums_latlong.parquet"
interim_df.to_parquet(interim_data_path, index=False)
print(f"\nInterim data saved to: {interim_data_path}")


Interim data saved to: ../data/interim/interim_museums_latlong.parquet


### 4.4. Verification

In [76]:
# Load the final interim data and verify its contents
try:
    df_verified = pd.read_parquet(interim_data_path)
    print(f"Verification successful. The following DataFrame is ready for analysis with {df_verified.shape[0]} samples and {df_verified.shape[1]} features:")
    display(df_verified.head())
except Exception as e:
    print(f"An error occurred while verifying the interim data: {e}")

Verification successful. The following DataFrame is ready for analysis with 1607 samples and 3 features:


,index,latitude,longitude
0,2,55.264551,-127.642812
1,3,45.963283,-66.641902
2,10,49.176354,-123.112783
3,15,49.261938,-123.151123
4,18,49.889559,-97.235744
